In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!apt-get update -qq
!apt-get install -y openjdk-11-jdk-headless -qq
!java -version

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
openjdk version "17.0.18" 2026-01-20
OpenJDK Runtime Environment (build 17.0.18+8-Ubuntu-122.04.1)
OpenJDK 64-Bit Server VM (build 17.0.18+8-Ubuntu-122.04.1, mixed mode, sharing)


In [3]:
!pip install pyspark --quiet

In [4]:
import os
import subprocess
import pyspark

result = subprocess.run(
    "java -XshowSettings:property -version 2>&1 | grep 'java.home'",
    shell=True, capture_output=True, text=True
)
java_home_path = result.stdout.strip().split("=")[-1].strip()
os.environ["JAVA_HOME"] = java_home_path
print(f"JAVA_HOME set to : {java_home_path}")
print(f"PySpark version  : {pyspark.__version__}")

JAVA_HOME set to : /usr/lib/jvm/java-17-openjdk-amd64
PySpark version  : 4.0.2


In [5]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("RandomForestModeling")
    .master("local[*]")
    .config("spark.driver.memory", "20g")
    .config("spark.sql.shuffle.partitions", "200")
    .getOrCreate()
)

spark.conf.set("spark.sql.parquet.int96RebaseModeInRead", "CORRECTED")
spark.conf.set("spark.sql.legacy.parquet.nanosAsLong", "true")
spark.sparkContext.setLogLevel("WARN")
spark

In [6]:
BASE_PATH = "/content/drive/MyDrive/OMDS Capstone/Data/flights_all_features_encoded"

TRAIN_PATH    = f"{BASE_PATH}/train.parquet"
VALIDATE_PATH = f"{BASE_PATH}/val.parquet"
TEST_PATH     = f"{BASE_PATH}/test.parquet"

# Avoid .count() — forces Spark to read all 31M rows into memory
train_df    = spark.read.parquet(TRAIN_PATH)
validate_df = spark.read.parquet(VALIDATE_PATH)

print("Data loaded.")

Data loaded.


In [7]:
print(train_df.columns)

['Year', 'Quarter', 'Month', 'DayofMonth', 'DayOfWeek', 'FlightDate', 'Reporting_Airline', 'Flight_Number_Reporting_Airline', 'Origin', 'Dest', 'CRSDepTime', 'DepTimeBlk', 'CRSArrTime', 'ArrDel15', 'CRSElapsedTime', 'Distance', 'DistanceGroup', 'date', 'dep_hour', 'arr_hour', 'dep_hour_minus2', 'arr_hour_minus2', 'origin_temp_f', 'origin_dewpoint_f', 'origin_humidity', 'origin_feels_like_f', 'origin_wind_kts', 'origin_gust_kts', 'origin_visibility', 'origin_precip_in', 'origin_wx_codes', 'origin_is_rain', 'origin_is_snow', 'origin_is_fog', 'origin_low_visibility', 'origin_high_wind', 'origin_severe_weather', 'dest_temp_f', 'dest_dewpoint_f', 'dest_humidity', 'dest_feels_like_f', 'dest_wind_kts', 'dest_gust_kts', 'dest_visibility', 'dest_precip_in', 'dest_wx_codes', 'dest_is_rain', 'dest_is_snow', 'dest_is_fog', 'dest_low_visibility', 'dest_high_wind', 'dest_severe_weather', 'is_weekend', 'is_holiday', 'origin_weather_missing', 'dest_weather_missing', 'carrier_delay_rate_30d', 'carrier_

In [8]:
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType

# Cast ArrDel15 to Double — Spark ML requires numeric label column
train_df    = train_df.withColumn("ArrDel15", F.col("ArrDel15").cast(DoubleType()))
validate_df = validate_df.withColumn("ArrDel15", F.col("ArrDel15").cast(DoubleType()))

print("ArrDel15 dtype:", dict(train_df.dtypes)["ArrDel15"])

ArrDel15 dtype: double


In [9]:
feature_cols = [
    "Month", "DayofMonth", "DayOfWeek", "dep_hour", "arr_hour",
    "CRSElapsedTime", "Distance", "DistanceGroup",
    "is_weekend", "is_holiday",
    "origin_delay_rate", "dest_delay_rate",
    "carrier_delay_rate_30d", "carrier_delay_rate_90d",
    "origin_delay_rate_30d", "origin_delay_rate_90d",
    "dest_delay_rate_30d", "dest_delay_rate_90d",
    "origin_departures_3h",
    "origin_temp_f", "origin_dewpoint_f", "origin_humidity",
    "origin_feels_like_f", "origin_wind_kts", "origin_gust_kts",
    "origin_visibility", "origin_precip_in",
    "origin_is_rain", "origin_is_snow", "origin_is_fog",
    "origin_low_visibility", "origin_high_wind", "origin_severe_weather",
    "dest_temp_f", "dest_dewpoint_f", "dest_humidity",
    "dest_feels_like_f", "dest_wind_kts", "dest_gust_kts",
    "dest_visibility", "dest_precip_in",
    "dest_is_rain", "dest_is_snow", "dest_is_fog",
    "dest_low_visibility", "dest_high_wind", "dest_severe_weather",
]

print(f"Total features: {len(feature_cols)}")

Total features: 47


In [10]:
from pyspark.ml.feature import VectorAssembler

# Sample BEFORE assembling — use 10% (~3.1M rows) to stay within memory limits
train_sample = train_df.sample(fraction=0.1, seed=42)

assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")

# No persist — avoid holding large DataFrames in memory simultaneously
train_sample = assembler.transform(train_sample)
validate_df  = assembler.transform(validate_df)

print("Assembler applied.")

Assembler applied.


In [11]:
from pyspark.ml.classification import RandomForestClassifier

rf = RandomForestClassifier(
    labelCol="ArrDel15",
    featuresCol="features",
    numTrees=20,
    maxDepth=5,
    subsamplingRate=0.7,
    seed=42
)

print("Training Random Forest...")
rf_model = rf.fit(train_sample)
print("Training complete.")

Training Random Forest...
Training complete.


In [12]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

predictions = rf_model.transform(validate_df)

auc = BinaryClassificationEvaluator(
    labelCol="ArrDel15", metricName="areaUnderROC"
).evaluate(predictions)

f1 = MulticlassClassificationEvaluator(
    labelCol="ArrDel15", metricName="f1"
).evaluate(predictions)

precision = MulticlassClassificationEvaluator(
    labelCol="ArrDel15", metricName="weightedPrecision"
).evaluate(predictions)

recall = MulticlassClassificationEvaluator(
    labelCol="ArrDel15", metricName="weightedRecall"
).evaluate(predictions)

print(f"Validation AUC      : {auc:.4f}")
print(f"Validation F1       : {f1:.4f}")
print(f"Validation Precision: {precision:.4f}")
print(f"Validation Recall   : {recall:.4f}")

Validation AUC      : 0.6654
Validation F1       : 0.7033
Validation Precision: 0.6310
Validation Recall   : 0.7944


In [13]:
# Feature importances — top 20
import pandas as pd

importances = rf_model.featureImportances
feat_imp_df = pd.DataFrame({
    "feature":    feature_cols,
    "importance": importances.toArray()
}).sort_values("importance", ascending=False)

print(feat_imp_df.head(20).to_string(index=False))

               feature  importance
   dest_delay_rate_30d    0.122303
 origin_severe_weather    0.109408
      origin_precip_in    0.105250
              arr_hour    0.097161
              dep_hour    0.090622
carrier_delay_rate_30d    0.079315
 origin_delay_rate_90d    0.062783
 origin_delay_rate_30d    0.059332
   origin_feels_like_f    0.056930
carrier_delay_rate_90d    0.048888
        origin_is_snow    0.048017
          dest_is_rain    0.026371
        dest_precip_in    0.023422
   dest_severe_weather    0.013473
 origin_low_visibility    0.008502
         dest_gust_kts    0.006915
  origin_departures_3h    0.006267
        origin_is_rain    0.006251
     origin_visibility    0.005579
     origin_dewpoint_f    0.005497
